# *Callbacks* y TensorBoard

Hay formas de tomar deciciones sobre el entrenamiento de un modelo mientras el mismo esta entrenando. De inspeccionar su estado y de tomar accion en base al mismo. Las opciones destacadas incluyen:
1. *Callbacks*
2. Tensorboard

# Callbacks

Los *callbacks* son objetos (intancias de una clase que implementan un metodo especifico) que pueden ser pasadas en la llamada a `fit` para ser invocados en ciertos puntos del entrenamiento.

Sus metodos pueden acceder a toda a informacion disponible sobre el estado de un modelo y su rendimiento. En base a eso pueden tomar acciones: interrumpir el entrenamiento, guardar el modelo, cargar un conjunto distinto de pesos, entre otras.

Algunos ejemplos de como se pueden usar los *callbacks* son:
- Puntos de guardado: guardar los pesos actuales del modelo en distintos puntos durante el entrenamiento.
- Detener temprano: interrumpir el entrenamiento cuando el error de validacion no mejora mas (guardando el mejor modelo obtenido durante el entrenamiento).
- Ajuste dinamico de parametros: como *learning rate* o *optimizer*.
- Informar metricas o visualizar representaciones: la barra de progreso de keras es un *callback*.

El modulo `keras.callbacks` incluye *callbaks* incorporados. A continuacion se muestran alunos que pueden servir para los ejemplos anteriores:
- `keras.callbacks.ModelCheckpoint`
- `keras.callbacks.EarlyStopping`
- `keras.callbacks.LearningRateScheduler`
- `keras.callbacks.ReduceLROnPlateau`
- `keras.callbacks.CSVLogger`

## `ModelCheckpoint` y `EarlyStopping`

A continuacion se muestra un ejemplo de como usar los callbacks `ModelCheckpoint` y `EarlyStopping` en un mismo entrenamiento. 

`EarlyStopping` se usa para detener el entrenamiento cuando una metrica no mejora durante un numero determinado de epocas. Esto permite detener la ejecucion tan pronto como se detecte que el modelo empieza a sobreentrenar.

`ModelCheckpoint` guarda el modelo durante el entrenamiento (opcionalmente solo el que tenga mejor rendimiento).

In [12]:
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import Dense
from keras import Sequential, Input
from keras.optimizers import RMSprop
from numpy.random import randint

# Lista de callbacks
callbacks_list = [
    # Monitorea accuracy en validacion y se detiene cuando no mejora por mas de una epoca (2)
    EarlyStopping(monitor='acc', patience=1),
    # Almacena solo el modelo que tenga menor error de validacion
    ModelCheckpoint(filepath='models/7.2.callbacks.keras', monitor='val_loss', save_best_only='True')
]

# Modelo
model = Sequential()
model.add(Input(shape = (10,)))
model.add(Dense(36, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Si se monitorea acc entonces esta debe ser parte de las metricas
model.compile(optimizer=RMSprop(), loss='binary_crossentropy', metrics=['acc'])

# Datos
samples = 1000
x = randint(0, 11, (samples, 10))
y = randint(0, 2, (samples, ))
x_val = randint(0, 11, (samples, 10))
y_val = randint(0, 2, (samples, ))

# Las callbacks monitorean el conjunto  de validacion por tanto es necesario especificar el conjunto
model.fit(
    x, 
    y,
    epochs=10,
    batch_size=32,
    callbacks=callbacks_list,
    validation_data=(x_val, y_val)
)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - acc: 0.5127 - loss: 1.4857 - val_acc: 0.5210 - val_loss: 0.7790
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.5093 - loss: 0.7837 - val_acc: 0.5380 - val_loss: 0.7265
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.5099 - loss: 0.7364 - val_acc: 0.5300 - val_loss: 0.7210


El entrenamiento se detubo de forma temprana al no mejorar el error de validacion luego de 1 epoca.

El mejor modelo fue guardado bajo el nombre *7.2.callbacks.keras* en la carpeta *models*.

## `ReduceLROnPlateau`

`ReduceLROnPlateau` se usa para modificar la tasa de aprendizaje cuando una metrica (ej: val_loss) deja de mejorar.

Es una herramienta efectiva para salir de minimos locales.

A continuacion se muestra un ejemplo donde se usa para reducir la tasa de aprendizaje cuando el error de validacion no mejora luego de cierto numero de epocas.

In [15]:
from keras.callbacks import ReduceLROnPlateau
from keras.layers import Dense
from keras import Sequential, Input
from keras.optimizers import RMSprop
from numpy.random import randint


callbacks_list = [
    # Si luego de 10 epocas el error de validacion no mejora entonces el learning rate se reduce a 1/10.
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=10)
]

# Modelo
model = Sequential()
model.add(Input(shape = (10,)))
model.add(Dense(36, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compilar
model.compile(optimizer=RMSprop(), loss='binary_crossentropy', metrics=['acc'])

# Datos
samples = 1000
x = randint(0, 11, (samples, 10))
y = randint(0, 2, (samples, ))
x_val = randint(0, 11, (samples, 10))
y_val = randint(0, 2, (samples, ))

# Las callbacks monitorean el conjunto  de validacion por tanto es necesario especificar el conjunto
model.fit(
    x, 
    y,
    epochs=50,
    batch_size=32,
    callbacks=callbacks_list,
    validation_data=(x_val, y_val)
)

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - acc: 0.5378 - loss: 0.9556 - val_acc: 0.4950 - val_loss: 0.8681 - learning_rate: 0.0010
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - acc: 0.5402 - loss: 0.8071 - val_acc: 0.5080 - val_loss: 0.7859 - learning_rate: 0.0010
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - acc: 0.5377 - loss: 0.7553 - val_acc: 0.4900 - val_loss: 0.7564 - learning_rate: 0.0010
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.5542 - loss: 0.7275 - val_acc: 0.5060 - val_loss: 0.7388 - learning_rate: 0.0010
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - acc: 0.5127 - loss: 0.7270 - val_acc: 0.5130 - val_loss: 0.7434 - learning_rate: 0.0010
Epoch 6/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.5482 - loss: 0.7105 - val_acc: 0.5100 - val_loss: 0.7235 - learning_rate: 0.0010
Epoch 7/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - acc: 0.5424 - loss: 0.6998 - val_acc: 0.5000 - val_loss: 0.7175 - learning_rate: 0.0010
Epoch 8/50
32/32 ━━

## Callback personalizada

Un usuario puede crear su propia *callback* creandola como subclase de `keras.callbacks.Callback`. Al crearla se pueden configurar cualquier cantidad  varias funciones, a continuacion se muestran algunos ejemplos.
- `on_epoch_begin`
- `on_epoch_end`
- `on_batch_begin`
- `on_batch_end`
- `on_train_beging`
- `on_train_end`

Cada metodo es llamado en cierto momento del entrenamiento, especificado por su nombre.

Los metodos son llamados un un agrgumento `logs`. Como un diccionario que contiene informacion de lotes, epocas o entrenamientos previos (por ejemplo metricas de validacion, de entrenamiento). Adicionalmente los *callbacks* pueden acceder `self.model` que es la instancia del modelo del cual esta siendo invocada la *callback*.

A continuacion se muestra un ejemplo de un callback personalizado que guarda en disco (como arreglos *Numpy*) las activaciones de cada capa del modelo al final de cada epoca. Computadas con la primera muestra del conjunto de validacion.

In [39]:
from keras.callbacks import Callback
from keras.models import Model
import numpy as np

class ActivationLogger(Callback):

    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data

    # Instanciar el modelo de activaciones.
    def on_train_begin(self, logs=None):
        # Modelo que retorna las activaciones de  cada capa
        layer_outputs = [layer.output for layer in self.model.layers]
        self.activations_model = Model(self.model.inputs, outputs=layer_outputs)
    
    def on_epoch_end(self, epoch, logs=None):
        if(self.validation_data is None):
            raise RuntimeError('Requires validation_data.')
        # Primera muestra de validacion
        validation_sample = self.validation_data[0][0:1]    
        # Recuperar activacion
        activations = self.activations_model.predict(validation_sample)
        # Almacenar resultado
        f = open('models/7.2.callback.activations_at_epoch_'+ str(epoch) + '.npz', 'wb')
        np.savez(f, **{f'layeer_{i}': act for i, act in enumerate(activations)})
        f.close()

A continuacion se muestra usando este *callback* se almacenan las activaciones de un modelo.

In [33]:
from keras.datasets import mnist

# Conjunto de datos MNIST
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

print('train_images', train_images.shape)
print('test_images', test_images.shape)


train_images (60000, 28, 28)
test_images (10000, 28, 28)


In [25]:
from keras.utils import to_categorical

# Formateado
conv_train_images = train_images.reshape((60000, 28, 28, 1))
conv_train_images = conv_train_images.astype('float32') / 255
conv_test_images = test_images.reshape((10000, 28, 28, 1))
conv_test_images = conv_test_images.astype('float32') / 255
conv_train_labels = to_categorical(train_labels)
conv_test_labels = to_categorical(test_labels)

print('conv_train_images', conv_train_images.shape)
print('conv_test_images', conv_test_images.shape)
print('conv_train_labels', conv_train_labels.shape)
print('conv_test_labels', conv_test_labels.shape)

conv_train_images (60000, 28, 28, 1)
conv_test_images (10000, 28, 28, 1)
conv_train_labels (60000, 10)
conv_test_labels (10000, 10)


In [30]:
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.models import Sequential
from keras import Input

# Modelo
model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(10, activation='softmax'))

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 3, 3, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,322 (364.54 KB)

 Trainable params: 93,322 (364.54 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
# Entrenar con callback
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['acc'])

callbacks_list = [
    ActivationLogger(validation_data=(conv_test_images, conv_test_labels))
]

model.fit(
    conv_train_images, 
    conv_train_labels, 
    callbacks=callbacks_list,
    epochs=5, 
    batch_size=64, 
    validation_data=(conv_test_images, conv_test_labels)
)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - acc: 0.9970 - loss: 0.0105WARNING:tensorflow:5 out of the last 9 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7f43d015e9e0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 602ms/step
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - acc: 0.9970 - loss: 0.0105 - val_acc: 0.9919 - val_loss: 0.0343
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/steptep - 

# TensorBoard

Es una herramienta basada en el navegador que permite viasualizar datos de interes durante y luego del entrenamiento de un modelo, como pueden ser:
- Metricas
- Arquitectura del modelo
- Histogramas de activacion y gradientes
- Exploracion de los embeddings

A continuacion se muestra el uso de la misma durante el entrenamiento de una ConvNet 1D para análisis de sentimientos en IMDB.

Solo se consideran las primeras 2,000 palabras en el vocabulario de IMDB para que los *embeddings* sean manejables.

In [2]:
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense, Input
from keras.optimizers import RMSprop
from keras.losses import BinaryCrossentropy
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Oculta la GPU para este proceso


max_features = 2000 # Palabras a considerar
max_len = 500       # Cortar textos luego de {max_len} palabras

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

model = Sequential([
    Input((max_len,)),
    Embedding(max_features, 128, name="embedding"),
    Conv1D(32, 7, activation='relu'),
    MaxPooling1D(5),
    Conv1D(32, 7, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(1)
])

model.summary()

loss = BinaryCrossentropy(from_logits=True)
model.compile(optimizer=RMSprop(), loss=loss, metrics=['acc'])

W0000 00:00:1778377795.829763  343164 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1778377795.880014  342710 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │       256,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 494, 32)        │        28,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 98, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 92, 32)         │         7,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 32)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 291,937 (1.11 MB)

 Trainable params: 291,937 (1.11 MB)

 Non-trainable params: 0 (0.00 B)

A continuacion se muestra un ejemplo de como decodificar una reseña del dataset/

In [3]:

encoder = imdb.get_word_index()
decoder = {value + 3: key for key, value in encoder.items()}
decoder[0] = "<PAD>"
decoder[1] = "<START>"
decoder[2] = "<UNK>"
decoder[3] = "<UNUSED>"
decoded_review = " ".join([decoder.get(i, "?") for i in x_train[2]])
print(decoded_review)

<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD

Antes de usar TensorBoard hay que crear un directorio donde almacenar los archivos de registro generados.

In [22]:
!mkdir tensorboard
!mkdir tensorboard/7.2.logdir

mkdir: cannot create directory ‘tensorboard’: File exists


TensorBoard se invoca por medio de un *callback* que escribe eventos de logs en la direccion especificada.

Para la visualizacion de embeddings se uso como guia el [tutorial practico de la pagina de Tensorflow](https://www.tensorflow.org/text/guide/word_embeddings?hl=es-419).

In [4]:
from keras.callbacks import TensorBoard, ModelCheckpoint

log_dir = "tensorboard/7.2.logdir"

callbacks = [
    TensorBoard(
        log_dir=log_dir, 
        #histogram_freq=1,   # Frecuencia (en epocas) para generar histogramas
        # embeddings_freq=1,  # Frecuencia (en epocas) para visualizar embeddings
        # embeddings_metadata={
        #     "embedding": "metadata.tsv"
        # },
        write_graph=True,
    )
]

history = model.fit(
    x_train, 
    y_train,
    batch_size=128,
    epochs=20,
    validation_split=0.2,
    callbacks=callbacks
)

Epoch 1/20


W0000 00:00:1778377806.784451  342710 cpu_allocator_impl.cc:82] Allocation of 40000000 exceeds 10% of free system memory.
W0000 00:00:1778377807.582775  343172 cpu_allocator_impl.cc:82] Allocation of 32768000 exceeds 10% of free system memory.
W0000 00:00:1778377807.635134  343169 cpu_allocator_impl.cc:82] Allocation of 30098432 exceeds 10% of free system memory.
W0000 00:00:1778377807.635178  343171 cpu_allocator_impl.cc:82] Allocation of 32768000 exceeds 10% of free system memory.
W0000 00:00:1778377807.714215  343171 cpu_allocator_impl.cc:82] Allocation of 32768000 exceeds 10% of free system memory.


157/157 ━━━━━━━━━━━━━━━━━━━━ 31s 191ms/step - acc: 0.5895 - loss: 0.6179 - val_acc: 0.8016 - val_loss: 0.4353
Epoch 2/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 26s 164ms/step - acc: 0.8173 - loss: 0.3892 - val_acc: 0.8358 - val_loss: 0.3494
Epoch 3/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 26s 164ms/step - acc: 0.8583 - loss: 0.3172 - val_acc: 0.8592 - val_loss: 0.3414
Epoch 4/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 26s 165ms/step - acc: 0.8791 - loss: 0.2726 - val_acc: 0.7886 - val_loss: 0.5953
Epoch 5/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 26s 166ms/step - acc: 0.8983 - loss: 0.2405 - val_acc: 0.8686 - val_loss: 0.3144
Epoch 6/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 26s 165ms/step - acc: 0.9141 - loss: 0.2080 - val_acc: 0.8176 - val_loss: 0.4083
Epoch 7/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 41s 166ms/step - acc: 0.9309 - loss: 0.1734 - val_acc: 0.8432 - val_loss: 0.3664
Epoch 8/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 27s 173ms/step - acc: 0.9434 - loss: 0.1470 - val_acc: 0.8508 - val_loss: 0.3728
Epoch 9/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 28s

## Visualizar embeddings

Con `get_layer` y `get_weights` se pueden obtener los pesos del modelo para la capa de embeddings.

Con get vocabulary se puede obtener el archivo de metadatos relacionado que muestra un token por linea.

In [18]:
weights = model.get_layer('embedding').get_weights()[0]
vocab = [decoder.get(i, "?") for i in range(max_features)]

print('Wheigths shape:', weights.shape)
print('Vocab shape:', len(vocab))
print('Vocab view:', vocab)

Wheigths shape: (2000, 128)
Vocab shape: 2000
Vocab view: ['<PAD>', '<START>', '<UNK>', '<UNUSED>', 'the', 'and', 'a', 'of', 'to', 'is', 'br', 'in', 'it', 'i', 'this', 'that', 'was', 'as', 'for', 'with', 'movie', 'but', 'film', 'on', 'not', 'you', 'are', 'his', 'have', 'he', 'be', 'one', 'all', 'at', 'by', 'an', 'they', 'who', 'so', 'from', 'like', 'her', 'or', 'just', 'about', "it's", 'out', 'has', 'if', 'some', 'there', 'what', 'good', 'more', 'when', 'very', 'up', 'no', 'time', 'she', 'even', 'my', 'would', 'which', 'only', 'story', 'really', 'see', 'their', 'had', 'can', 'were', 'me', 'well', 'than', 'we', 'much', 'been', 'bad', 'get', 'will', 'do', 'also', 'into', 'people', 'other', 'first', 'great', 'because', 'how', 'him', 'most', "don't", 'made', 'its', 'then', 'way', 'make', 'them', 'too', 'could', 'any', 'movies', 'after', 'think', 'characters', 'watch', 'two', 'films', 'character', 'seen', 'many', 'being', 'life', 'plot', 'never', 'acting', 'little', 'best', 'love', 'over', 

Para usar el Proyector de embeddings es necesario cargar 2 archivos distintos.
- Un archivo de vectores (informacion de los embeddings).
- Un archivo de palabras (traduccion de cada embedding).

Ambos en formato separado por tabulaciones.

In [19]:
import io
import os

out_v = io.open(os.path.join(log_dir, 'vectors.tsv'), 'w', encoding='utf-8')
out_m = io.open(os.path.join(log_dir, 'metadata.tsv'), 'w', encoding='utf-8')

for index, word in enumerate(vocab):
    # if(index==0):
    #     continue # <PAD>
    vec = weights[index]
    out_v.write("\t".join([str(x) for x in vec]) + "\n")
    out_m.write(word + "\n")

out_m.close()
out_v.close()
    

Con estos 2 archivos se puede realizar una visualizacion de los embeddings y busqueda desde el [proyector online de Tensorboard](https://projector.tensorflow.org/?hl=es-419&_gl=1*1twgo3q*_ga*MTk4Mzc0MTI0Mi4xNzUyNTI3NjQx*_ga_W0YLR4190T*czE3NzMxMDM5NzMkbzY4JGcxJHQxNzczMTA1Mzg0JGo2MCRsMCRoMA..).

Otra opcion es visualizar los embeddings mediante el proyector integrado en tensorflow. Esto se puede hacer de 2 formas.
1. Cargando los archivos manualmente tal como en la version online.
2. Enlazando los archivos desde la etapa de configuracion.

En los callbacks de arriba se incluye el codigo necesario para visualizarlo, no obstante puede que nuevas versiones de tensorflow no lo soporten.

Se puede levantar el servidor de TensorBoard mediante la linea de comandos.

```
tensorboard --logdir=tensorboard/7.2.logdir
```

Por defecto se expone en `http://localhost:6006`

## Grafico del modelo

Las ultimas versiones de *TensorBoard* no soportan el uso del flag `write_graph=True` que activa la generacion de graficos. 

Existen otras opciones para visualizar el modelo que es descargar su imagen como tal. A continuacion se muestra un ejemplo de como hacerlo.

In [28]:
# Para usar 'plot_model' es necesario ademas de las librerias python que las variables de entorno esten configuradas.
!apt-get update
!apt-get install -y graphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fontconfig fontconfig-config fonts-dejavu-core fonts-liberation libann0
  libbsd0 libcairo2 libcdt5 libcgraph6 libdatrie1 libdeflate0 libfontconfig1
  libfreetype6 libfribidi0 libgd3 libgraphite2-3 libgts-0.7-5 libgts-bin
  libgvc6 libgvpr2 libharfbuzz0b libice6 libjbig0 libjpeg-turbo8 libjpeg8
  liblab-gamut1 libltdl7 libmd0 libpango-1.0-0 libpangocairo-1.0-0
  libpangoft2-1.0-0 libpathplan4 libpixman-1-0 libpng16-16 libsm6 libthai-data
  libthai0 libtiff5 libwebp7 libx11-6 libx11-data libxau6 libxaw7
  libxcb-render0 libxcb-shm0 libxcb1 libxdmcp6 libxext6 libxmu6 libxpm4
  libxrender1 libxt6 x11-common
Suggested packages:
  gsfonts graphviz-doc libgd-tools
The following NEW packages will be installed:
  fontconfig fontconfig-config fonts-dejavu-core fonts-liberation graphviz
  libann0 libbsd0 libcairo2 libcdt5 libcgraph6 libdatrie1 li

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(
    model,
    to_file="images/7.2.model-graph.png"
)

<img src="images/7.2.model-graph.png" style="width:150px;">

La funcion `plot_models` tiene varios parametros que permiten enriquecer las visualizaciones. A continuacion se muestra un ejemplo de como incluir las dimensiones relacionadas a cada capa.

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(
    model,
    to_file="images/7.2.model-graph-with-shapes.png",
    show_shapes=True
)

<img src="images/7.2.model-graph-with-shapes.png" style="width:250px;">